In [7]:
import numpy as np

from weather.config import (
    Experiment,
    WeatherFixedParams,
    WeatherGridParams,
    MLPFixedParams,
    MLPGridParams,
    FitFixedParams,
    FitGridParams,
)

from weather.search import Search

from mlp.utils import (
    plot_loss,
    regression_report,
    classification_report_binary,
    plot_roc_auc,
    plot_accuracy,
    accuracy_within_tolerance,
)


In [8]:
SEED = 42
np.random.seed(SEED)


In [9]:
# =========================================================
# 1) TEMPERATURE REGRESSION
# =========================================================
exp_temp_encoding = Experiment(
    name="temperature_regression_encoding",

    # =========================
    # WEATHER
    # =========================
    weather_fixed=WeatherFixedParams(
        target="temperature",
        target_mode="regression",
        # target_threshold=6.0,
        data_dir="../data",
        skip_day=True,
        # normalization="global",
        encode_wind_direction=True,
    ),

    weather_grid=WeatherGridParams(
        window_aggregation="flatten",
        window_size=[3],
        normalization="standardize",
        input_variables=[
            ("temperature",),
            # ("temperature", "humidity"),
            # ("temperature", "humidity", "pressure"),
            # ("temperature", "humidity", "pressure", "wind_speed"),
            ("temperature", "humidity", "pressure", "wind_speed", "wind_direction"),
        ],
        aggregations={
            "temperature": ("mean", "min", "max"),
            "humidity": ("mean", "min", "max"),
            "pressure": ("mean", "min", "max"),
            "wind_speed": ("mean", "max"),
            "wind_direction": ("mean",),
        },
        cities=[
            # ("Vancouver",),
            ("Vancouver", "Seattle", "Portland"),
            ("Beersheba", "Tel Aviv District", "Eilat", "Haifa", "Nahariyya", "Jerusalem")
        ]
    ),

    # =========================
    # MLP
    # =========================
    mlp_fixed=MLPFixedParams(
        task="regression",
        beta = 0.9,
        beta2 = 0.999,
        eps = 1e-8,
        adaptive_lr=True,
        lr_decay=0.99,
    ),

    mlp_grid=MLPGridParams(
        hidden_layers = [
            (16, 64),
            (32, 64),
            (20, 40),
            (128, 64),
            (100, 128, 64),
        ],
        loss="mae",
        activation="gelu",
        learning_rate=0.01,
        seed=SEED,
        use_bias=True,
        optimizer="momentum",
    ),

    # =========================
    # FIT
    # =========================
    fit_fixed=FitFixedParams(
        verbose=True,
        log_every=None,
        use_tqdm=True,
        one_hot_if_needed=True,
        early_stopping=True,
        patience=100,
        min_delta=0.0001,
    ),

    fit_grid=FitGridParams(
        epochs=400,
        batch_size="auto",
        shuffle=False,
        val_split=0.1,
    ),
)


In [ ]:
search = Search()

results1 = search.run(exp_temp_encoding)



Starting experiment: temperature_regression_encoding

Building dataset
  → TRAIN split


Vancouver | windows: 100%|██████████| 1518/1518 [00:02<00:00, 590.99it/s]
Seattle | windows: 100%|██████████| 1518/1518 [00:02<00:00, 625.89it/s]
Portland | windows: 100%|██████████| 1518/1518 [00:02<00:00, 633.75it/s]


  → TEST split


Vancouver | windows: 100%|██████████| 361/361 [00:00<00:00, 728.93it/s]
Seattle | windows: 100%|██████████| 361/361 [00:00<00:00, 566.97it/s]
Portland | windows: 100%|██████████| 361/361 [00:00<00:00, 557.72it/s]



Configuration run 1/20:
WEATHER (variable):
  - input_variables: ('temperature',)
  - cities: ('Vancouver', 'Seattle', 'Portland')
  - window_size: 3
MLP (variable):
  - hidden_layers: (16, 64)

Training model


Training:  37%|███▋      | 149/400 [00:13<00:22, 11.37it/s, acc=n/a, loss=2.2541, lr=0.00223689]


Early stopping at epoch 150, best val_loss=2.293746 after 100 epochs without improvement.
Training finished in 13.11 seconds

Configuration run 2/20:
WEATHER (variable):
  - input_variables: ('temperature',)
  - cities: ('Vancouver', 'Seattle', 'Portland')
  - window_size: 3
MLP (variable):
  - hidden_layers: (32, 64)

Training model


Training:  36%|███▌      | 144/400 [00:17<00:46,  5.54it/s, acc=n/a, loss=2.3917, lr=0.00237593]

In [ ]:
from IPython.core.display import HTML

for run in results1:
    model = run["model"]
    y_test = run["y_test"]
    y_pred = run["y_pred"]
    y_proba = run["y_proba"]

    if model.task == "binary":
        raise ValueError("Expected regression task, got binary.")

    history = run["history"]
    accuracy_history = run["accuracy_history"]
    config_log = run.get("config_log", {})

    metrics = regression_report(
        y_true=y_test,
        y_pred=y_pred,
    )

    acc_2 = accuracy_within_tolerance(y_test, y_pred, tol=2.0)
    acc_25 = accuracy_within_tolerance(y_test, y_pred, tol=2.5)

    if acc_2 <= 0.634:
        continue

    print("\n\n" + "=" * 80)
    print(f"EXPERIMENT: {run['experiment']}")
    print(f"TASK: {model.task.upper()}")
    print("=" * 80)

    # ========= CONFIGURATION (VARIABLE PARAMS) =========
    if config_log:
        print("CONFIGURATION (variable params):")
        for section, params in config_log.items():
            if not params:
                continue
            print(f"  {section.upper()}:")
            for k, v in params.items():
                print(f"    - {k}: {v}")
        print("-" * 80)

    # ===================== METRICS =====================


    if acc_2 >= 0.66:
        color = "#2e7d32"   # dark green
    elif acc_2 >= 0.65:
        color = "#558b2f"   # olive green
    elif acc_2 >= 0.64:
        color = "#f9a825"   # amber
    elif acc_2 >= 0.634:
        color = "#ef6c00"   # orange
    else:
        color = "#c62828"   # red

    print("=== TEST METRICS (REGRESSION) ===")
    print(f"MAE              : {metrics['mae']:.4f}")
    print(f"MSE              : {metrics['mse']:.4f}")
    print(f"RMSE             : {metrics['rmse']:.4f}")
    display(HTML(
        f"""
        <div style="
            font-family: 'JetBrains Mono', 'Consolas', 'Menlo', monospace;
            font-size: 13px;
            color: {color};
            padding-left: 12px;
            margin: 4px 0;
        ">
            <b>Accuracy |err|≤2°C</b>: {acc_2:.4f}
        </div>
        """
    ))
    print(f"Accuracy |err|≤2°C   : {acc_2:.4f}")

    # --- plots ---
    # plot_loss(
    #     history,
    #     title="Train Loss Evolution",
    # )
    #
    # if model.task != "regression":
    #     if accuracy_history and not all(np.isnan(accuracy_history)):
    #         plot_accuracy(
    #             accuracy_history,
    #             title="Accuracy evolution",
    #         )
    #
    # # ROC only for binary classification
    # if model.task == "binary":
    #     plot_roc_auc(
    #         y_true=y_test,
    #         y_score=y_proba,
    #         title="ROC Curve",
    #     )
